In [1]:
from pathlib import Path
import yaml

import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import transforms

import matplotlib.pyplot as plt

from tqdm import tqdm
from kornia.losses import FocalLoss

import timm

import numpy as np

# Moduli progetto
from models.model import ChimeraSeg
from models.decoder import Decoder, DecoderWithFAM
from training.train import Trainer
from training.losses import PrototypicalGlobalLocalTripletLoss 
from training.metrics import MeanIoU
from data.streethazards import StreetHazards, STREET_HAZARDS_CLASSES, get_classes_as_dict
from data.multi_epochs_dataloader import MultiEpochsDataLoader
from utils.misc import get_device, fix_random, print_summary

from utils.visualize import COLORS, color

# Configurazioni di ambiente
%load_ext autoreload
%autoreload 2

/home/aarcara/miniconda3/envs/ml4cv/lib/python3.11/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.4' (you have '2.0.3'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()



Premessa: come il mio precedente notebook, link al notebook, sono interessato ad avere delle reti efficienti che mi permettano di poter trainare liberamente e fare diverse prove anche su un dispositivo mac con `mps`.

Guardando i diversi paper che ottenevano alti score nel benchmark SegmentMeIfYouCan, i migliori come mIoU utilizzano Mask classification e di conseguenza Mask2Former con l'aggiunta di un modulo o di tecniche per aggiungere anche il rilevamento di anomalie. Tra tutti ho preferito implementare il paper `Open Semantic Segmentation with Class Similarity` che ottiene un mIoU competitivo, rimanendo al contempo veloce a differenza degli approcci con Mask2Former più lenti di natura, e con un rilevamento di anomalie decisamente migliore. Successivamente, data la recente uscita del paper `Open Panoptic Segmentation`, sempre degli stessi autori, che oltre ad aggiungere la Panoptic Segmentation, risolve alcuni problemi del precedente approccio.

TODO

* Insert anomalies in validation set
* mAP on anomalies
* open mIoU

In [2]:
config_path = "./config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

fix_random(config['seed'])
device = get_device()

🚀 CUDA device is available!


In [2]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
imgH = config['training']['img_height']
imgW = config['training']['img_width']

data_transforms = {
    "train": A.Compose([
        A.RandomCrop(height=imgH, width=imgW),
        A.HorizontalFlip(p=0.5),
        #A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.CoarseDropout(max_holes=8, max_height=imgH//8, max_width=imgW//8, min_holes=1, min_height=imgH//16, min_width=imgW//16, p=0.5),
        A.Normalize(mean=imagenet_mean, std=imagenet_std, max_pixel_value=255.0),
        ToTensorV2(),
    ]),
    "val": A.Compose([
        A.Resize(height=imgH, width=imgW),
        A.Normalize(mean=imagenet_mean, std=imagenet_std, max_pixel_value=255.0),
        ToTensorV2(),
    ])
}

class Denormalize(object):
    def __init__(self, mean, std):
        self.denormalize = A.Compose([
            A.Normalize(
                mean=tuple(-m / s for m, s in zip(mean, std)),
                std=tuple(1.0 / s for s in std),
                max_pixel_value=1.0,
            ),
            A.FromFloat(max_value=255, dtype="uint8"),
        ])
        
    def __call__(self, img):
        if isinstance(img, torch.Tensor):
            img = np.transpose(img.cpu().detach().numpy().squeeze(), (1, 2, 0))
        return self.denormalize(image=img)['image']
    
denorm = Denormalize(mean=imagenet_mean, std=imagenet_std)


NameError: name 'config' is not defined

In [3]:
train_root = Path.home() / config['paths']['train_path']

train_data = StreetHazards(
    train_root,
    "training",
    transforms=data_transforms["train"]
)

val_data = StreetHazards(
    train_root,
    "validation",
    transforms=data_transforms["val"]
)

val_data_with_anomalies()

num_classes = len(STREET_HAZARDS_CLASSES)
class_dict = get_classes_as_dict()

print(f"Number of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(val_data)}")
print(f"Number of classes: {num_classes}")

NameError: name 'Path' is not defined

In [4]:
from torchvision.utils import make_grid

def visualize_augmentations(dataset, num_samples=4):
    img, _ = dataset[0]
    imgs = [img]

    for _ in range(num_samples - 1):
        img_np = img.permute(1, 2, 0).numpy()
        augmented = data_transforms['train'](image=img_np)['image']
        #imgs.append(torch.from_numpy(augmented).permute(2, 0, 1))

    img_grid = make_grid(imgs, nrow=4, normalize=True)

    plt.figure(figsize=(15,15))
    plt.imshow(img_grid.permute(1, 2, 0).numpy())
    plt.title("Original and Augmented Images", fontsize=16)
    plt.axis('off')
    plt.show()

visualize_augmentations(train_data)

NameError: name 'train_data' is not defined

In [5]:
import os
NUM_WORKERS = os.cpu_count() - 1

train_loader = MultiEpochsDataLoader(
    train_data,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=3
)

val_loader = MultiEpochsDataLoader(
    val_data,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

NameError: name 'MultiEpochsDataLoader' is not defined

In [7]:
class_weights = train_data.get_class_weights().to(device)
class_weights = class_weights[:-1]
num_classes = num_classes - 1
print(len(class_weights))
print(num_classes)

13
13


COMMENTI:

* la wce converge meglio rispetto alla focal loss, ma risulta meno stabile e ogni tanto sul validation diverge decisamente
* best score right now on validation it's 0.55 mIoU,
* attualmente il mio modello fa fatica sulle parti lontane dell'immagine dove il modello non riesce a capire esattamente cosa c'è nell'immagine e sugli oggetti molto piccoli
* il modello non è preciso ai bordi, come affrontare il problema?
* aspp fa qualcosa?



In [10]:
from training.abl import ABL 
from pytorch_ood.detector import WeightedEBO
from pytorch_ood.loss import VOSRegLoss

phi = torch.nn.Linear(1, 2).to(device)
weights_energy = torch.nn.Linear(num_classes, 1).to(device)
torch.nn.init.uniform_(weights_energy.weight)


do_train = False

model_name = 'resnet18d'
d = 128 
fpn_features = [64, 64, 128, 512]
atrous_rates = (4, 8, 12)
encoder = timm.create_model(model_name, features_only=True, pretrained=True, out_indices=(0, 1, 2, 4), output_stride=16)
#decoder = Decoder([128, 256, 512], (360, 640), num_classes, atrous_rates=(6, 12, 18))
decoder = DecoderWithFAM(fpn_features, num_classes, input_size=(imgH, imgW), d=d, atrous_rates=atrous_rates)
model = ChimeraSeg(encoder, decoder)
#model(torch.rand((4, 3, 640, 360))
#print_summary(model, (config['training']['batch_size'], 3, imgH, imgW))

#criterions = [(0.9, nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config['training']['ls'])), (0.1, OWLoss(num_classes))]
#criterions = [(1.0, prototypicalgloballocaltripletloss(num_classes=num_classes, margin_global=1.5, margin_local=0.5)), (1.0, nn.crossentropyloss(weight=class_weights))]
#criterions = [(1.0, PrototypicalGlobalLocalTripletLoss(num_classes=num_classes, margin_global=1.5, margin_local=0.5, magnitude=config['training']['anchors_magnitude'])), 
#            (1.0, FocalLoss(alpha=1, gamma=2, reduction='mean'))]
criterions = [(1.0, VOSRegLoss(phi, weights_energy, device=device))]
# criterions = [(1.0, PrototypicalGlobalLocalTripletLoss(num_classes=num_classes, margin_global=1.5, margin_local=0.5, magnitude=config['training']['anchors_magnitude'])), 
#  #             (1.0, nn.CrossEntropyLoss(weight=class_weights)),
#               (1.0, ABL())
#               ]
#criterions = [(0.5, nn.CrossEntropyLoss(weight=class_weights))]
#criterions = [(1.0, nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config['training']['ls']))]
#criterions = [(20, FocalLoss(weight=class_weights, alpha=0.5, gamma=2, reduction='mean')), (1, DiceLoss())]

loss_str = ",".join([l.__class__.__name__ for _, l in criterions])
atrous_str = "-".join(map(str, atrous_rates))

trainer = Trainer(
    config, 
    model, 
    device, 
    train_loader, 
    criterions,
    val_loader,
    class_dict=class_dict, 
    metrics=[MeanIoU(num_classes)],
    use_amp=True
)

from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = (
    f"ENC_{model_name}"
    f"_DEC_{len(fpn_features) - 1}FAM{d}"
    f"_ASPP{atrous_str}"
    f"_IMG{imgW}x{imgH}"
    f"_LOSS_{loss_str}"
    f"_{timestamp}"
)

if do_train:
    trainer.train(run_name)
else:
    model.load_state_dict(torch.load("saved/models/model_epoch50_miou0.6518.pt", weights_only=True))

INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/resnet18d.ra2_in1k)
INFO:timm.models._hub:[timm/resnet18d.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:timm.models._builder:Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.


In [9]:
weights_energy

Linear(in_features=13, out_features=1, bias=True)

In [10]:
model.eval()

ChimeraSeg(
  (encoder): FeatureListNet(
    (conv1): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=T

In [11]:
test_root = Path.home() / config['paths']['test_path']

test_data = StreetHazards(
    test_root,
    "test",
    transforms=data_transforms["val"]
)

test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=config['training']['batch_size'],
    shuffle=False
    #num_workers=NUM_WORKERS,
    #pin_memory=True,
    #persistent_workers=True,
    #prefetch_factor=2
)

In [75]:
from pytorch_ood.augment.img import InsertCOCO

coco_transform = InsertCOCO(
    coco_dir="data/coco",
    exclude_classes="Streethazards",
    p=1,
)


In [12]:
import ipywidgets as widgets

anchors = torch.eye(num_classes, device='cuda') * config['training']['anchors_magnitude']
print(anchors)

def anomaly_map(logits: torch.Tensor, threshold: float):
    """
    Computes L2 distances between each pixel embedding and the anchors,
    """
    B, C, H, W = logits.shape
    print(logits.device)

    logits_reshaped = logits.permute(0, 2, 3, 1).reshape(B, -1, num_classes)

    distance_function = torch.nn.PairwiseDistance(p=2)
    
    distances = torch.zeros(B, H*W, num_classes, device='cuda')
    
    for i in range(num_classes):
        anchor = anchors[i].view(1, 1, -1)
        distances[:, :, i] = distance_function(logits_reshaped, anchor)
    
    min_distances, closest_class = torch.min(distances, dim=2)
    anomaly_mask = (min_distances > threshold).reshape(B, H, W)

    print(min_distances)
    print(min_distances.shape)
    
    return anomaly_mask 

def visualize_predictions(model, dataset, device, threshold=0.5):
    def show_prediction(idx, threshold_value):
        """Funzione interna per visualizzare una singola predizione"""
        img, mask = dataset[idx]
        
        img = img.unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(img)[1]
            pred = torch.argmax(logits, dim=1)
            anomaly = anomaly_map(logits, threshold_value)
        
        plt.figure(figsize=(15, 10))
        
        plt.subplot(231)
        plt.imshow(denorm(img))
        plt.title('Image')
        plt.axis('off')
        
        plt.subplot(232)
        plt.imshow(color(mask.squeeze(), COLORS))
        plt.title('Ground Truth')
        plt.axis('off')
        
        plt.subplot(233)
        plt.imshow(color(pred.cpu().squeeze(), COLORS))
        #plt.imshow(pred.cpu().squeeze())
        plt.title('Predicted')
        plt.axis('off')
        
        # plt.subplot(234)
        # plt.imshow(anomaly.cpu().squeeze(), cmap='hot')
        # plt.title('Anomaly Mask')
        # plt.axis('off')

        plt.tight_layout()
        plt.show()
    
    idx_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(dataset)-1,
        step=1,
        description='Indice:'
    )
    
    threshold_slider = widgets.FloatSlider(
        value=threshold,
        min=0.,
        max=10.,
        step=0.1,
        description='Threshold:'
    )
    
    widgets.interact(show_prediction, idx=idx_slider, threshold_value=threshold_slider)

tensor([[20.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0., 20.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0., 20.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0., 20.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0., 20.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0., 20.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0., 20.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0., 20.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 20.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 20.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 20.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 20.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 20.]],
       device='cuda:0')


In [13]:
visualize_predictions(model, test_data, 'cuda')

interactive(children=(IntSlider(value=0, description='Indice:', max=1499), FloatSlider(value=0.5, description=…

In [51]:
import numpy as np

## steps per finire il progetto

- secondo validation set injectato con "anomalie"
- calcolo mean average precision

In [ ]:
!pip install pytorch-ood 

5